# Lab 4 — Why It Matters: Retrieval Quality IS Answer Quality (ES|QL edition)

**Thesis:** In a RAG system, the model is not the ceiling — **retrieval is**. Same model + worse retrieval = worse answer. *"The model didn't get dumber. The retrieval got worse."*

This ES|QL edition has a headline you can't do in the retriever DSL: **the entire RAG pipeline — hybrid retrieval, reranking, and LLM answer synthesis — as ONE ES|QL query.** `FORK` + `FUSE` retrieve, `RERANK` sharpens, `COMPLETION` calls the LLM. No orchestration code.

## What you'll learn
- The one-query RAG pipeline: `FORK | FUSE | RERANK | COMPLETION`
- The proof: GOOD vs BAD retrieval, same model + same question, answers side by side
- `COMPLETION` requires the **`completion`** task type (not `chat_completion` streaming)
- Agent Builder finale: the same FORK+FUSE retriever, registered as a multi-hop agent tool

## Interface
This lab is **notebook-primary** (COMPLETION makes one LLM call per row — fine in a notebook, slow as a Discover spinner). There's an optional **Discover "peek"** at the end so you can watch the whole pipeline run in the database UI once.

## Before you start
- **In Instruqt:** `ES_ENDPOINT`, `ES_API_KEY`, and `KIBANA_URL` are pre-configured.
- **Repo:** `export ES_ENDPOINT=...`, `export ES_API_KEY=...`, `export KIBANA_URL=https://...kb...:443`

In [ ]:
# --- Workshop helpers (inline — same block across all ES|QL notebooks) ---
# ES|QL edition: every search runs through es.esql.query() instead of es.search().
# Defined inline so this notebook is self-contained and runs from the repo too.

import os, json, time
import requests
from elasticsearch import Elasticsearch

INDEX = "aiewf-workshop-docs"

ES_ENDPOINT = os.environ.get("ES_ENDPOINT")
ES_API_KEY  = os.environ.get("ES_API_KEY")
if not ES_ENDPOINT or not ES_API_KEY:
    raise ValueError(
        "Set ES_ENDPOINT and ES_API_KEY.\n"
        "  In Instruqt: pre-configured in the sandbox.\n"
        "  Re-running the repo: export ES_ENDPOINT=https://...  export ES_API_KEY=..."
    )

# request_timeout=120: RERANK and COMPLETION (Labs 4-5) call inference per row and
# can take several seconds — the default 10s would time out the LLM step.
es = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY, request_timeout=120)

def esql(query, **params):
    """Run an ES|QL query with named parameters (?name in the query string).

    Usage:  esql(QUERY, q="securing cluster traffic")
    ES|QL named params take the form params=[{"name": value}, ...]. If your pinned
    client rejects named params, switch to positional `?` and params=[value, ...] —
    never f-string the query text in (injection + teaches the wrong pattern).
    """
    param_list = [{k: v} for k, v in params.items()] if params else None
    return es.esql.query(query=query, params=param_list, format="json")

def rows(resp):
    """Turn an ES|QL response ({columns, values}) into a list of dicts keyed by column."""
    cols = [c["name"] for c in resp["columns"]]
    return [dict(zip(cols, vals)) for vals in resp["values"]]

def show_esql(resp, fields=("id", "title", "summary"), score=True):
    """Pretty-print ES|QL rows as a ranked table (mirrors the DSL notebooks' show_hits)."""
    data = rows(resp)
    if not data:
        print("  (no rows)"); return
    for rank, r in enumerate(data, 1):
        cols = "  ".join(str(r.get(f, "")) for f in fields)
        sc = r.get("_score")
        s = f"  {sc:.4f}" if score and sc is not None else ""
        print(f"  #{rank:<2}{s}  {cols}")

print("✓ ES|QL helpers loaded")


# The completion endpoint COMPLETION will call. MUST be a `completion` task type —
# NOT `.anthropic-...-chat_completion` (that one is streaming-only and ES|QL's
# COMPLETION command cannot use it).
COMPLETION_ID = ".anthropic-claude-4.5-haiku-completion"

print("✓ Lab 4 helpers loaded")
print(f"  COMPLETION endpoint: {COMPLETION_ID}")

In [ ]:
# Verify the completion-task endpoint exists on THIS project before we rely on it.
eps = es.inference.get().body.get("endpoints", [])
completion_eps = [e["inference_id"] for e in eps if e.get("task_type") == "completion"]
print("completion-task endpoints available:")
for e in completion_eps:
    print(f"  {e}")
if COMPLETION_ID not in completion_eps:
    print(f"\n⚠ {COMPLETION_ID} not found. Pick another completion endpoint above and set")
    print("  COMPLETION_ID accordingly, or the COMPLETION cells will fail.")
else:
    print(f"\n✓ {COMPLETION_ID} is available.")

## The one-query RAG pipeline

Here is a full RAG pipeline as a single ES|QL statement. Read it as stages:

```esql
FROM aiewf-workshop-docs METADATA _score, _id, _index
| FORK ( WHERE MATCH(body, ?q)          | SORT _score DESC | LIMIT 50 )   -- BM25 recall
       ( WHERE MATCH(body_semantic, ?q) | SORT _score DESC | LIMIT 50 )   -- semantic recall
| FUSE                                                                     -- RRF fuse
| LIMIT 20
| RERANK ?q ON body WITH {"inference_id": ".jina-reranker-v3"}            -- precision
| LIMIT 1                                                                  -- best doc only
| EVAL prompt = CONCAT(
    "Answer the question using ONLY the document below.\n",
    "Title: ", title, "\n\n", body, "\n\nQuestion: ", ?q)             -- build the prompt
| COMPLETION answer = prompt WITH {"inference_id": <completion endpoint>} -- call the LLM
| KEEP id, title, answer
```

`FORK`/`FUSE` get strong recall, `RERANK` reorders the top 20 for precision, `LIMIT 1` takes the single best doc, `EVAL` builds the prompt string from that doc's fields, and `COMPLETION` sends it to the LLM and returns the generated `answer` column. We build it up in stages first.

In [ ]:
# Stage 1+2 — FORK + FUSE + RERANK (no LLM yet). See what doc the pipeline would feed the model.
question = "How does Index Lifecycle Management move data through hot, warm, and cold phases?"

Q_RETRIEVE = (
    "FROM aiewf-workshop-docs METADATA _score, _id, _index\n"
    "| FORK ( WHERE MATCH(body, ?q)          | SORT _score DESC | LIMIT 50 )\n"
    "       ( WHERE MATCH(body_semantic, ?q) | SORT _score DESC | LIMIT 50 )\n"
    "| FUSE | SORT _score DESC | LIMIT 20\n"
    '| RERANK ?q ON body WITH {"inference_id": ".jina-reranker-v3"}\n'
    "| SORT _score DESC | LIMIT 5 | KEEP id, title, _score"
)
print(f"Question: {question!r}\nTop docs after FORK|FUSE|RERANK:\n")
show_esql(esql(Q_RETRIEVE, q=question), fields=("id", "title", "_score"))

In [ ]:
# Full pipeline — add EVAL prompt + COMPLETION. One query: retrieve -> fuse -> rerank -> generate.
Q_RAG = (
    "FROM aiewf-workshop-docs METADATA _score, _id, _index\n"
    "| FORK ( WHERE MATCH(body, ?q)          | SORT _score DESC | LIMIT 50 )\n"
    "       ( WHERE MATCH(body_semantic, ?q) | SORT _score DESC | LIMIT 50 )\n"
    "| FUSE | SORT _score DESC | LIMIT 20\n"
    '| RERANK ?q ON body WITH {"inference_id": ".jina-reranker-v3"}\n'
    "| SORT _score DESC | LIMIT 1\n"
    '| EVAL prompt = CONCAT("Answer the question using ONLY the document below. '
    'Title: ", title, " === Document: ", body, " === Question: ", ?q)\n'
    '| COMPLETION answer = prompt WITH {"inference_id": "%s"}\n'
    "| KEEP id, title, answer"
) % COMPLETION_ID

print(f"Question: {question!r}\n(running the full FORK|FUSE|RERANK|COMPLETION pipeline — ~5-15s)\n")
res = rows(esql(Q_RAG, q=question))[0]
print(f"Source doc: [{res['id']}] {res['title']}\n")
print("=== LLM ANSWER (generated inside Elasticsearch) ===")
print(res["answer"])

## The experiment: GOOD vs BAD retrieval — same model, same question

The pipeline above produced a good answer because retrieval found the right doc. Now hold **everything** constant — same question, same `COMPLETION` model — and change **only the retrieval**. We force the FORK branches to draw from off-topic documents (`trap_type == "version-specific"` — release notes, nothing to do with SAML) and watch the same model produce a useless answer.

In [ ]:
def run_rag(question, bad=False):
    """One-query RAG. bad=True constrains FORK to off-topic (version-specific) docs."""
    filt = 'WHERE trap_type == "version-specific" AND ' if bad else "WHERE "
    q = (
        "FROM aiewf-workshop-docs METADATA _score, _id, _index\n"
        f"| FORK ( {filt}MATCH(body, ?q)          | SORT _score DESC | LIMIT 50 )\n"
        f"       ( {filt}MATCH(body_semantic, ?q) | SORT _score DESC | LIMIT 50 )\n"
        "| FUSE | SORT _score DESC | LIMIT 1\n"
        '| EVAL prompt = CONCAT("Answer the question using ONLY the document below. '
        'If it lacks the answer, say you do not have enough information. '
        'Title: ", title, " === Document: ", body, " === Question: ", ?q)\n'
        '| COMPLETION answer = prompt WITH {"inference_id": "%s"}\n'
        "| KEEP id, title, answer"
    ) % COMPLETION_ID
    return rows(esql(q, q=question))[0]

question = "How does Index Lifecycle Management move data through hot, warm, and cold phases?"
good = run_rag(question, bad=False)
print(f"GOOD CONTEXT — retrieved [{good['id']}] {good['title']}\n")
print("=== GOOD ANSWER ===")
print(good["answer"])

In [ ]:
bad = run_rag(question, bad=True)
print(f"BAD CONTEXT — retrieval forced to off-topic [{bad['id']}] {bad['title']}\n")
print("=== BAD ANSWER ===")
print(bad["answer"])

In [ ]:
print(f"Question: {question!r}")
print("=" * 70)
print("GOOD CONTEXT ANSWER:\n" + "-" * 70)
print(good["answer"])
print("\nBAD CONTEXT ANSWER:\n" + "-" * 70)
print(bad["answer"])
print("\n" + "=" * 70)
print("\n💡 The model didn't get dumber. The retrieval got worse.")
print("   Same model, same question — only the FORK filter changed. This is not a")
print("   model-quality problem. It's a retrieval-quality problem.")

## A note on that filter — it is NOT access control

We used `WHERE trap_type == "version-specific"` to *shape* what the model saw. It's tempting to think the same trick enforces security ("only return docs this user may see"). **It does not.** An app-side `WHERE` filter is a *relevance* control, not an *authorization* control — anyone who can change the query can remove it.

Real access control is enforced at the **credential** level, not in the query: Role-Based Access Control (RBAC) and **Document-Level Security (DLS)**, where the role descriptor itself carries a query that the user can never bypass:

```python
es.security.create_api_key(
    name="contractor-key",
    role_descriptors={"restricted_reader": {"indices": [{
        "names": ["aiewf-workshop-docs"],
        "privileges": ["read"],
        "query": {"bool": {"must_not": {"term": {"confidential": True}}}}  # enforced server-side
    }]}},
)
```
With DLS, the `confidential` docs are invisible to that key no matter what ES|QL the holder writes. Shown as reference (not run live — the sandbox key is restricted).

## Finale — the same retriever, as a multi-hop Agent Builder agent

The one-query pipeline is great when *you* write the query. But an **agent** decides on its own what to search, reads the results, and runs a *second* search when the first doesn't fully answer. Agent Builder runs that loop server-side.

The agent's tool is the **exact same `FORK ... | FUSE` hybrid retriever** you've used all workshop — registered via the Kibana Agent Builder API. The next cell creates the tool, a "Diagnose and Fix" skill, and the agent (idempotent — safe to re-run). It's the same setup the sandbox ran at boot.

In [ ]:
# Create the Agent Builder hybrid-search tool + Diagnose and Fix skill + multi-hop agent (idempotent).
KIBANA_URL = os.environ.get("KIBANA_URL")
if not KIBANA_URL:
    raise ValueError(
        "Set KIBANA_URL — the Kibana endpoint (.kb.), not Elasticsearch (.es.).\n"
        "  In Instruqt: pre-configured. Repo: export KIBANA_URL=https://...kb...:443"
    )

AB_TOOL_ID  = "search-workshop-docs-hybrid"
AB_SKILL_ID = "workshop-docs-diagnose-fix"
AB_AGENT_ID = "workshop-docs-agent"

# The Lab 3 RRF hybrid retriever, in ES|QL (FORK = two arms, FUSE = RRF). Same query
# the one-shot pipeline above used for recall — here it's the agent's tool.
HYBRID_ESQL = (
    "FROM aiewf-workshop-docs METADATA _score, _id, _index\n"
    "| FORK ( WHERE match(body, ?query) | SORT _score DESC | LIMIT 50 )\n"
    "       ( WHERE match(body_semantic, ?query) | SORT _score DESC | LIMIT 50 )\n"
    "| FUSE\n"
    "| SORT _score DESC\n"
    "| KEEP id, title, url, body, _score\n"
    "| LIMIT 5"
)

SKILL_CONTENT = """# Diagnose and Fix playbook

When the user reports an Elasticsearch symptom and wants both *why it happens* and *how to fix it*, work in this order:

1. **Find the cause.** Search the symptom (the error code, the cluster state, the failure) with `search-workshop-docs-hybrid` to identify the root cause.
2. **Find the fix.** The first results usually point at a specific API, setting, or subsystem. Run a SECOND search targeting that fix — the exact setting name, the repair API, the config key.
3. **Answer as Cause -> Fix -> Citations.**
   - **Cause:** one or two sentences on what's actually wrong.
   - **Fix:** the concrete steps, naming the exact settings, commands, or API calls from the docs.
   - **Citations:** the doc titles you used, as [title].

Never guess a setting name or a command — if the docs don't contain it, say so."""

AGENT_INSTRUCTIONS = """You are an Elasticsearch documentation assistant built on hybrid retrieval.

## Your tool
- **search-workshop-docs-hybrid**: hybrid (BM25 + semantic, RRF-fused) search over the Elasticsearch docs corpus. It returns the top 5 docs with id, title, url, and body.

## How you work (multi-hop)
1. Call `search-workshop-docs-hybrid` with the user's question to get an initial set of docs.
2. Read the results. If they fully answer the question, write the answer.
3. If answering well requires a fact the first results point to but don't fully cover (a setting, a root cause, a related subsystem), run a SECOND search with a refined query targeting that specific gap, then answer from the combined results. Prefer one focused follow-up over many.
4. Ground every claim in retrieved docs. Cite sources as [title]. If the docs don't contain the answer, say so — do not guess.

## Style
Concise and technical. Use short sections or bullets. Name the specific settings, codes, or commands the docs mention."""

def _ab(method, path, body=None):
    resp = requests.request(method, f"{KIBANA_URL}{path}",
        headers={"Authorization": f"ApiKey {ES_API_KEY}", "kbn-xsrf": "true",
                 "Content-Type": "application/json"}, json=body, timeout=60)
    try:    return resp.status_code, resp.json()
    except ValueError: return resp.status_code, resp.text

# Idempotent reset — delete in dependency order: agent -> skill -> tool.
_ab("DELETE", f"/api/agent_builder/agents/{AB_AGENT_ID}")
_ab("DELETE", f"/api/agent_builder/skills/{AB_SKILL_ID}")
_ab("DELETE", f"/api/agent_builder/tools/{AB_TOOL_ID}")

status, resp = _ab("POST", "/api/agent_builder/tools", {
    "id": AB_TOOL_ID, "type": "esql",
    "description": ("Hybrid (BM25 + semantic) search over the Elasticsearch documentation "
        "corpus, fused with RRF. Use this to find relevant docs for ANY question about "
        "Elasticsearch. Returns id, title, url, and body for the top 5 docs."),
    "tags": ["workshop", "search", "hybrid", "rag"],
    "configuration": {"query": HYBRID_ESQL,
        "params": {"query": {"type": "string",
                             "description": "The natural-language search query or keywords."}}},
})
print(f"{'✓' if status == 200 else '✗'} tool  '{AB_TOOL_ID}'  (HTTP {status})")

status, resp = _ab("POST", "/api/agent_builder/skills", {
    "id": AB_SKILL_ID, "name": "Diagnose and Fix",
    "description": ("Use when the user reports an Elasticsearch symptom and wants both the "
        "cause and the fix. Drives a two-search diagnose-then-repair flow."),
    "content": SKILL_CONTENT, "tool_ids": [AB_TOOL_ID],
})
skill_ok = status == 200
print(f"{'✓' if skill_ok else '⚠'} skill '{AB_SKILL_ID}'  (HTTP {status})" + ("" if skill_ok else f" — skipped: {resp}"))

agent_cfg = {"instructions": AGENT_INSTRUCTIONS, "tools": [{"tool_ids": [AB_TOOL_ID]}]}
if skill_ok:
    agent_cfg["skill_ids"] = [AB_SKILL_ID]
agent_body = {"id": AB_AGENT_ID, "name": "Workshop Docs Agent",
    "description": "Multi-hop RAG agent over the workshop docs corpus, powered by hybrid retrieval.",
    "labels": ["workshop"], "configuration": agent_cfg}
status, resp = _ab("POST", "/api/agent_builder/agents", agent_body)
if status != 200 and "skill_ids" in agent_cfg:
    print(f"⚠ agent create with skill_ids failed (HTTP {status}); retrying without it...")
    agent_cfg.pop("skill_ids", None)
    status, resp = _ab("POST", "/api/agent_builder/agents", agent_body)
print(f"{'✓' if status == 200 else '✗'} agent '{AB_AGENT_ID}'  (HTTP {status})")
print(f"\nOpen Agent Builder in Kibana: {KIBANA_URL}/app/agent_builder")

In [ ]:
# Run the agent via the converse API and watch it multi-hop.
agent_question = (
    "My Elasticsearch container keeps dying with exit code 137. Why does that happen, "
    "and what specific JVM and memory settings should I change to prevent it?"
)
print(f"Asking the agent: {agent_question!r}\n(full agent loop runs server-side; ~15-25s)\n")

status, result = _ab("POST", "/api/agent_builder/converse",
                     {"agent_id": AB_AGENT_ID, "input": agent_question})
if status != 200:
    raise RuntimeError(f"converse failed (HTTP {status}): {result}")

retrieval_hops = 0
for step in result.get("steps", []):
    if step.get("type") == "reasoning":
        print(f"  💭 {step['reasoning'][:140]}")
    elif step.get("type") == "tool_call":
        params = step.get("params", {}) or {}
        if "query" in params:
            retrieval_hops += 1
            print(f"  🔧 retrieval hop {retrieval_hops}: {step.get('tool_id','')}  query={params['query']!r}")
        elif "skill" in params:
            print(f"  🧩 {step.get('tool_id','')}: loaded skill {params['skill']!r}")

resp = result.get("response", "")
answer = resp if isinstance(resp, str) else (resp.get("message") if isinstance(resp, dict)
         else "".join(b.get("text", "") for b in resp if isinstance(b, dict)))
print(f"\n{'='*60}")
print(f"Retrieval hops: {retrieval_hops}   |   LLM calls: {result.get('model_usage', {}).get('llm_calls', '?')}")
print(f"{'='*60}\n=== AGENT ANSWER ===")
print(answer)

## (Optional) The Discover "peek" — watch the whole pipeline run in the database UI

Open the **Kibana Discover** tab, switch the query bar to **ES|QL**, and paste the full pipeline. It runs end to end in the UI — retrieval, rerank, *and* the LLM call — and the generated answer lands in a single `answer` cell.

```esql
FROM aiewf-workshop-docs METADATA _score, _id, _index
| FORK ( WHERE MATCH(body, "How does Index Lifecycle Management move data through hot, warm, and cold phases?")          | SORT _score DESC | LIMIT 50 )
       ( WHERE MATCH(body_semantic, "How does Index Lifecycle Management move data through hot, warm, and cold phases?") | SORT _score DESC | LIMIT 50 )
| FUSE | SORT _score DESC | LIMIT 20
| RERANK "How does Index Lifecycle Management move data through hot, warm, and cold phases?" ON body WITH {"inference_id": ".jina-reranker-v3"}
| SORT _score DESC | LIMIT 1
| EVAL prompt = CONCAT("Answer the question using ONLY the document below. Title: ", title, " === Document: ", body, " === Question: How does Index Lifecycle Management move data through hot, warm, and cold phases?")
| COMPLETION answer = prompt WITH {"inference_id": ".anthropic-claude-4.5-haiku-completion"}
| KEEP id, title, answer
```

> ⏱️ **Expect a multi-second spinner** — Discover waits on the reranker *and* the LLM call. That latency is exactly why this lab is notebook-primary; the peek is to *see* that a complete RAG pipeline is now a database query. Expand the `answer` cell to read the full generated text.

---

## Closing

You expressed an entire RAG pipeline — hybrid retrieval, reranking, LLM synthesis — as **one ES|QL query**, then ran the *same* retriever as a multi-hop agent. The query language changed from the retriever DSL; the thesis didn't:

> **Retrieval quality, not model quality, determines answer quality.**

---
*Bonus: open `lab5-esql-reranking.ipynb` to go deeper on the `RERANK` stage.*